# GPAGD Demo: Poisson 1D
This notebook demonstrates the Geometric Physics‑Aware Gradient Descent (GPAGD) optimizer on a simple Poisson 1D PDE.

In [ ]:
!pip install git+https://github.com/YOUR_USERNAME/GPAGD-Optimizer.git

In [ ]:
import torch
import torch.nn as nn
from gpagd import GeometricPhysicsGD
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Poisson 1D problem
class Poisson1D:
    def __init__(self, n_colloc=1000):
        self.x = torch.linspace(0, 1, n_colloc).reshape(-1,1)
    def residual(self, model):
        x = self.x.requires_grad_(True)
        u = model(x)
        u_x = torch.autograd.grad(u, x, grad_outputs=torch.ones_like(u), create_graph=True)[0]
        u_xx = torch.autograd.grad(u_x, x, grad_outputs=torch.ones_like(u_x), create_graph=True)[0]
        f = torch.sin(2*np.pi*x)
        r = -u_xx - f
        return r.pow(2).mean()

problem = Poisson1D()
model = nn.Sequential(nn.Linear(1,50), nn.Tanh(), nn.Linear(50,50), nn.Tanh(), nn.Linear(50,1))
optimizer = GeometricPhysicsGD(model.parameters(), lr=1e-3, rho=0.1, alpha=1.0)

# Dummy projector and noise for this demo (simplified)
def projector(g): return g
def noise(): return torch.tensor(0.1)
def noise_level(): return 0.1
def physics_residual(): return problem.residual(model)

losses = []
for epoch in range(1000):
    def closure():
        optimizer.zero_grad()
        loss = problem.residual(model)
        loss.backward()
        return loss
    loss = optimizer.step(closure, projector, physics_residual, noise, 1000, noise_level)
    losses.append(loss.item())
    if epoch % 200 == 0:
        print(f"Epoch {epoch}, loss = {loss.item():.4e}")

plt.semilogy(losses)
plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.title('GPAGD on Poisson 1D'); plt.grid(True)
plt.show()